In [ ]:
import findspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

findspark.init()
spark = SparkSession.builder.master("local[*]").getOrCreate()
df = spark.read.csv('train.csv', inferSchema=True, header=True)

In [56]:
df.head(10)

[Row(PassengerId=1, Survived=0, Pclass=3, Name='Braund, Mr. Owen Harris', Sex='male', Age=22.0, SibSp=1, Parch=0, Ticket='A/5 21171', Fare=7.25, Cabin=None, Embarked='S'),
 Row(PassengerId=2, Survived=1, Pclass=1, Name='Cumings, Mrs. John Bradley (Florence Briggs Thayer)', Sex='female', Age=38.0, SibSp=1, Parch=0, Ticket='PC 17599', Fare=71.2833, Cabin='C85', Embarked='C'),
 Row(PassengerId=3, Survived=1, Pclass=3, Name='Heikkinen, Miss. Laina', Sex='female', Age=26.0, SibSp=0, Parch=0, Ticket='STON/O2. 3101282', Fare=7.925, Cabin=None, Embarked='S'),
 Row(PassengerId=4, Survived=1, Pclass=1, Name='Futrelle, Mrs. Jacques Heath (Lily May Peel)', Sex='female', Age=35.0, SibSp=1, Parch=0, Ticket='113803', Fare=53.1, Cabin='C123', Embarked='S'),
 Row(PassengerId=5, Survived=0, Pclass=3, Name='Allen, Mr. William Henry', Sex='male', Age=35.0, SibSp=0, Parch=0, Ticket='373450', Fare=8.05, Cabin=None, Embarked='S'),
 Row(PassengerId=6, Survived=0, Pclass=3, Name='Moran, Mr. James', Sex='male',

In [57]:
df.count()
df = df.drop("Cabin")
df = df.drop("PassengerId")
df = df.drop("Name")
df = df.drop("Ticket")
df = df.drop("Embarked")
df = df.drop("Fare")

In [58]:
df = df.dropna()
df.count()

714

In [59]:
df.head(10)

[Row(Survived=0, Pclass=3, Sex='male', Age=22.0, SibSp=1, Parch=0),
 Row(Survived=1, Pclass=1, Sex='female', Age=38.0, SibSp=1, Parch=0),
 Row(Survived=1, Pclass=3, Sex='female', Age=26.0, SibSp=0, Parch=0),
 Row(Survived=1, Pclass=1, Sex='female', Age=35.0, SibSp=1, Parch=0),
 Row(Survived=0, Pclass=3, Sex='male', Age=35.0, SibSp=0, Parch=0),
 Row(Survived=0, Pclass=1, Sex='male', Age=54.0, SibSp=0, Parch=0),
 Row(Survived=0, Pclass=3, Sex='male', Age=2.0, SibSp=3, Parch=1),
 Row(Survived=1, Pclass=3, Sex='female', Age=27.0, SibSp=0, Parch=2),
 Row(Survived=1, Pclass=2, Sex='female', Age=14.0, SibSp=1, Parch=0),
 Row(Survived=1, Pclass=3, Sex='female', Age=4.0, SibSp=1, Parch=1)]

In [60]:
stringIndexer = StringIndexer(inputCol="Sex", outputCol="Sex_index", stringOrderType="frequencyDesc")
df = stringIndexer.fit(df).transform(df)
df.show(10)

+--------+------+------+----+-----+-----+---------+
|Survived|Pclass|   Sex| Age|SibSp|Parch|Sex_index|
+--------+------+------+----+-----+-----+---------+
|       0|     3|  male|22.0|    1|    0|      0.0|
|       1|     1|female|38.0|    1|    0|      1.0|
|       1|     3|female|26.0|    0|    0|      1.0|
|       1|     1|female|35.0|    1|    0|      1.0|
|       0|     3|  male|35.0|    0|    0|      0.0|
|       0|     1|  male|54.0|    0|    0|      0.0|
|       0|     3|  male| 2.0|    3|    1|      0.0|
|       1|     3|female|27.0|    0|    2|      1.0|
|       1|     2|female|14.0|    1|    0|      1.0|
|       1|     3|female| 4.0|    1|    1|      1.0|
+--------+------+------+----+-----+-----+---------+
only showing top 10 rows


In [61]:
assembler = VectorAssembler(inputCols=["Pclass","Age", "SibSp", "Parch", "Sex_index"], outputCol='features')
output = assembler.transform(df)
output.show(5)

+--------+------+------+----+-----+-----+---------+--------------------+
|Survived|Pclass|   Sex| Age|SibSp|Parch|Sex_index|            features|
+--------+------+------+----+-----+-----+---------+--------------------+
|       0|     3|  male|22.0|    1|    0|      0.0|[3.0,22.0,1.0,0.0...|
|       1|     1|female|38.0|    1|    0|      1.0|[1.0,38.0,1.0,0.0...|
|       1|     3|female|26.0|    0|    0|      1.0|[3.0,26.0,0.0,0.0...|
|       1|     1|female|35.0|    1|    0|      1.0|[1.0,35.0,1.0,0.0...|
|       0|     3|  male|35.0|    0|    0|      0.0|[3.0,35.0,0.0,0.0...|
+--------+------+------+----+-----+-----+---------+--------------------+
only showing top 5 rows


In [62]:
train, test = output.randomSplit([0.8, 0.2])

In [63]:
lr = LogisticRegression(featuresCol='features', labelCol='Survived')
model = lr.fit(train)
predictions = model.transform(train)

In [64]:
evaluator = MulticlassClassificationEvaluator(labelCol='Survived')
print('Evaluation:', evaluator.evaluate(predictions))

Evaluation: 0.7928538201746271
